# Lab 2 Exercise

The REST walkthrough already showed the building blocks. In this exercise, those building blocks are provided so you can focus on detecting failure and rerouting.

> **Run cells in order from top to bottom.** Later exercises depend on variables set by earlier cells. Jumping ahead will cause `NameError` or stale state.

<details>
<summary>New to Jupyter? Click here</summary>

- **Run a cell**: click the cell, then press **Shift+Enter** (or click the ▶ Run button in the toolbar)
- **Cell state**: a number like `[3]` means the cell has run; `[*]` means it is still running
- **Cells share state**: variables set in one cell are available in all cells below it — this is why order matters
- **Stop a long-running cell**: click the **■ Stop** button in the toolbar, or go to **Kernel → Interrupt**
- **Something broke**: go to **Kernel → Restart & Clear Output**, then re-run every cell from the top

</details>

## What this exercise is about

In the walkthrough, ONOS was still helping you because `org.onosproject.fwd` could install forwarding rules for new traffic.

In this exercise, you will turn that app off and replace that behavior with your own notebook code.

Your notebook will act like a tiny controller app:

1. install rules for the current path between `h1` and `h2`
2. watch the links on that path
3. detect when one of those links fails
4. remove the old rules and install new ones on the alternate path

<img src="triangle-reroute.svg" alt="Automatic reroute overview" width="620">

## Before you start

Keep Mininet, ONOS CLI, and Jupyter running.

In the ONOS CLI, run:

```text
onos> app deactivate org.onosproject.fwd
```

Then check that the old rules are gone:

```text
onos> flows
```

Wait until no `org.onosproject.fwd` rules remain.

In the Mininet CLI, confirm connectivity is now broken:

```text
mininet> h1 ping -c 1 h2
```

That ping should fail. If it does not, the `fwd` app is still active — deactivate it and wait a few seconds before retrying.

In [ ]:
import json
import requests
import time

## Provided code

Run the next cell. It defines all the helper functions you will use below.

You do not need to read or understand the internals — just run it and use the function names when the exercises ask you to.

In [ ]:
# REST helpers: talk to ONOS over HTTP and return parsed results.

def api_get(endpoint, base, auth):
    response = requests.get(f'{base}/{endpoint}', auth=auth, timeout=5)
    response.raise_for_status()
    return response.json()


def api_post(endpoint, data, base, auth):
    response = requests.post(
        f'{base}/{endpoint}',
        auth=auth,
        json=data,
        timeout=5,
    )
    response.raise_for_status()
    return response


def api_delete(endpoint, base, auth):
    response = requests.delete(f'{base}/{endpoint}', auth=auth, timeout=5)
    response.raise_for_status()
    return response


# Topology helpers: move from host IPs to the switches where those hosts attach.

def find_host_by_ip(hosts, ip):
    for host in hosts:
        if ip in host.get('ipAddresses', []):
            return host
    return None


def get_host_location(host):
    locations = host.get('locations', [])
    if not locations:
        return None, None
    location = locations[0]
    return location['elementId'], location['port']


def get_path(src_device, dst_device, base, auth):
    data = api_get(f'paths/{src_device}/{dst_device}', base, auth)
    paths = data.get('paths', [])
    if not paths:
        return None
    return paths[0]['links']


# Flow-rule helpers: build the JSON body ONOS expects for one IPv4 rule.

def build_flow_rule(src_ip, dst_ip, out_port, app_id, priority=40000):
    return {
        'priority': priority,
        'timeout': 0,
        'isPermanent': True,
        'appId': app_id,
        'treatment': {
            'instructions': [
                {'type': 'OUTPUT', 'port': str(out_port)}
            ]
        },
        'selector': {
            'criteria': [
                {'type': 'ETH_TYPE', 'ethType': '0x0800'},
                {'type': 'IPV4_SRC', 'ip': f'{src_ip}/32'},
                {'type': 'IPV4_DST', 'ip': f'{dst_ip}/32'},
            ]
        },
    }


def install_path_rules(
    path_links,
    src_ip,
    dst_ip,
    src_host_port,
    dst_host_port,
    base,
    auth,
    app_id,
):
    # The path API gives inter-switch links, so we add host-facing edge rules too.
    first_device = path_links[0]['src']['device']
    last_device = path_links[-1]['dst']['device']

    for link in path_links:
        src_device = link['src']['device']
        src_port = link['src']['port']
        dst_device = link['dst']['device']
        dst_port = link['dst']['port']

        forward_rule = build_flow_rule(src_ip, dst_ip, src_port, app_id)
        reverse_rule = build_flow_rule(dst_ip, src_ip, dst_port, app_id)

        api_post(f'flows/{src_device}', forward_rule, base, auth)
        api_post(f'flows/{dst_device}', reverse_rule, base, auth)

    dst_edge_rule = build_flow_rule(src_ip, dst_ip, dst_host_port, app_id)
    src_edge_rule = build_flow_rule(dst_ip, src_ip, src_host_port, app_id)

    api_post(f'flows/{last_device}', dst_edge_rule, base, auth)
    api_post(f'flows/{first_device}', src_edge_rule, base, auth)


# Cleanup helper: remove only the rules that belong to this notebook's app_id.

def remove_rules_by_app_id(device_ids, base, auth, app_id):
    removed = 0
    for device_id in device_ids:
        flows = api_get(f'flows/{device_id}', base, auth).get('flows', [])
        for flow in flows:
            if flow.get('appId') == app_id:
                api_delete(f"flows/{device_id}/{flow['id']}", base, auth)
                removed += 1
    return removed


# Port-status helper: read the ONOS view of one switch's ports.

def get_port_status(device_id, base, auth):
    return api_get(f'devices/{device_id}/ports', base, auth)['ports']

## Ready check

Run the next cell before doing anything else.

You want to see:

- at least 2 hosts discovered
- 0 remaining `org.onosproject.fwd` rules
- `[ready]` at the end

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

devices = api_get('devices', base, auth).get('devices', [])
hosts = api_get('hosts', base, auth).get('hosts', [])

fwd_rule_count = 0
for device in devices:
    flows = api_get(f"flows/{device['id']}", base, auth).get('flows', [])
    fwd_rule_count += sum(1 for flow in flows if flow.get('appId') == 'org.onosproject.fwd')

print(f'devices discovered: {len(devices)}')
print(f'hosts discovered: {len(hosts)}')
print(f'org.onosproject.fwd rules remaining: {fwd_rule_count}')

if len(hosts) >= 2 and fwd_rule_count == 0:
    print('[ready] You can continue.')
else:
    print('[not ready] Fix the checks above first.')

## Sanity check

Before you build rerouting, make sure the provided helpers can install the current path once.

Run the next cell.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'
app_id = 'org.onosproject.rest'

device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
remove_rules_by_app_id(device_ids, base, auth, app_id)

hosts = api_get('hosts', base, auth)['hosts']
src_host = find_host_by_ip(hosts, src_ip)
dst_host = find_host_by_ip(hosts, dst_ip)
src_device, src_host_port = get_host_location(src_host)
dst_device, dst_host_port = get_host_location(dst_host)
path_links = get_path(src_device, dst_device, base, auth)

if not path_links:
    print('No path found. Recheck ONOS discovery before continuing.')
else:
    install_path_rules(
        path_links,
        src_ip,
        dst_ip,
        src_host_port,
        dst_host_port,
        base,
        auth,
        app_id,
    )
    print(json.dumps(path_links, indent=2))
    print('Rules installed for the current path.')

**Verify**

In the Mininet CLI, run:

```text
mininet> h1 ping -c 3 h2
```

That ping should work — our code installed the rules.

In the ONOS CLI, run:

```text
onos> flows
```

You should see rules with `appId=org.onosproject.rest`.

## Part 1: Find the switches on the current path

Complete `get_active_devices`.

**Why this matters**

Before rerouting, we need to know which switches have our old rules on them so we can clean them up. This function builds that set.

**Goal**

Return the set of device IDs touched by the current path.

**Hints**

- each link has a `src` object and a `dst` object — look inside each for the device ID
- a `set()` avoids duplicates if the same switch appears on more than one link

In [ ]:
def get_active_devices(path_links):
    devices = set()

    for link in path_links:
        # TODO: add the source switch from this link
        # TODO: add the destination switch from this link
        pass

    return devices

<details>
<summary>Show solution</summary>

```python
def get_active_devices(path_links):
    devices = set()

    for link in path_links:
        devices.add(link['src']['device'])
        devices.add(link['dst']['device'])

    return devices
```

</details>

**Verify**

Run the next cell. You should see a list of switch IDs — one per switch on the current path.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

hosts = api_get('hosts', base, auth)['hosts']
src_host = find_host_by_ip(hosts, '10.0.0.1')
dst_host = find_host_by_ip(hosts, '10.0.0.2')
src_device, _ = get_host_location(src_host)
dst_device, _ = get_host_location(dst_host)
path_links = get_path(src_device, dst_device, base, auth)

print(sorted(get_active_devices(path_links)))

## Part 2: Detect whether the current path has failed

Complete `detect_failure`.

**Why this matters**

The monitor loop calls this every few seconds. It needs to notice the moment a port on our path goes down so rerouting can start immediately.

**Goal**

Return `True` if any port along the current path is down. Otherwise return `False`.

**Hints**

- `device_id` and `path_port` are already set above the TODO — use them
- `get_port_status` returns a list; each item has a `port` field and an `isEnabled` field
- port numbers from the API can come back as integers — cast to `str()` before comparing with `path_port`
- check `isEnabled` to know whether the port is up or down

In [ ]:
def detect_failure(path_links, base, auth):
    for link in path_links:
        device_id = link['src']['device']
        path_port = str(link['src']['port'])
        ports = get_port_status(device_id, base, auth)

        for port in ports:
            # TODO: return True when you find the matching path port and it is disabled
            pass

    return False

<details>
<summary>Show solution</summary>

```python
def detect_failure(path_links, base, auth):
    for link in path_links:
        device_id = link['src']['device']
        path_port = str(link['src']['port'])
        ports = get_port_status(device_id, base, auth)

        for port in ports:
            if str(port['port']) == path_port and not port['isEnabled']:
                return True

    return False
```

</details>

**Verify**

Run the next cell to save a snapshot of the current (healthy) path. **Do this before bringing the link down.**

You should see the path links printed as JSON. The variable `path_links_before_failure` is used by parts 3 and 4.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

hosts = api_get('hosts', base, auth)['hosts']
src_host = find_host_by_ip(hosts, '10.0.0.1')
dst_host = find_host_by_ip(hosts, '10.0.0.2')
src_device, _ = get_host_location(src_host)
dst_device, _ = get_host_location(dst_host)
path_links_before_failure = get_path(src_device, dst_device, base, auth)
print(json.dumps(path_links_before_failure, indent=2))

Now go to the Mininet CLI and run:

```text
mininet> link s1 s2 down
```

Then run the next cell. It should print `True`.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
print(detect_failure(path_links_before_failure, base, auth))

## Part 3: Reroute once after a failure

Complete `reroute_once`.

**Why this matters**

This is the core of a controller application: when a link fails, clean up the stale rules, ask ONOS for a new path, and install fresh rules on it. Everything else is just monitoring that calls this function.

**Goal**

Remove the old rules, query the new path, install new rules, and return the new path.

**Note** Keep the `s1-s2` link down from part 2. If you accidentally brought it back up, run `link s1 s2 down` in Mininet again before running the verify cell.

**Hints**

- both TODOs are single function calls — look at the helper functions in the Provided code cell; all the variables you need are already in scope above each TODO
- the rest of the logic (fetching hosts, computing the new path, handling the no-path case) is already written for you

In [ ]:
def reroute_once(path_links, src_ip, dst_ip, base, auth, app_id):
    device_ids = sorted(get_active_devices(path_links))
    # TODO: remove the old rules from the switches in device_ids
    time.sleep(2)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    new_path_links = get_path(src_device, dst_device, base, auth)

    if not new_path_links:
        return None

    # TODO: install rules for new_path_links
    return new_path_links

<details>
<summary>Show solution</summary>

```python
def reroute_once(path_links, src_ip, dst_ip, base, auth, app_id):
    device_ids = sorted(get_active_devices(path_links))
    remove_rules_by_app_id(device_ids, base, auth, app_id)
    time.sleep(2)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    new_path_links = get_path(src_device, dst_device, base, auth)

    if not new_path_links:
        return None

    install_path_rules(
        new_path_links,
        src_ip,
        dst_ip,
        src_host_port,
        dst_host_port,
        base,
        auth,
        app_id,
    )
    return new_path_links
```

</details>

**Verify**

The `s1-s2` link should still be down from the previous step.

Run the next cell. It will call `reroute_once` and print the new path.

Then in Mininet run:

```text
mininet> h1 ping -c 3 h2
```

That ping should work again — traffic is now going via the alternate hop. In the ONOS CLI:

```text
onos> flows
```

You should see rules with `appId=org.onosproject.rest`.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'
app_id = 'org.onosproject.rest'

new_path_links = reroute_once(path_links_before_failure, src_ip, dst_ip, base, auth, app_id)
print(json.dumps(new_path_links, indent=2))

## Part 4: Build the automatic loop

Complete `monitor_and_reroute`.

**Why this matters**

`reroute_once` handles a single failure. This function wraps it in a poll loop so the app keeps watching and reacts to any future failure automatically — exactly what a real controller application does.

**Goal**

Install the current path, poll every few seconds, and reroute automatically when a failure appears.

**The one thing to fix**

After calling `reroute_once(...)`, the return value (the new path) must be saved back into `path_links`. If you don't, the loop will keep watching the old (broken) path and think the link is still down on every iteration.

Uncomment the `path_links = reroute_once(...)` line in the `if detect_failure` branch.

In [ ]:
def monitor_and_reroute(src_ip, dst_ip, base, auth, app_id, poll_interval=5):
    device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
    remove_rules_by_app_id(device_ids, base, auth, app_id)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    path_links = get_path(src_device, dst_device, base, auth)

    if not path_links:
        print('No path found.')
        return

    install_path_rules(
        path_links,
        src_ip,
        dst_ip,
        src_host_port,
        dst_host_port,
        base,
        auth,
        app_id,
    )
    print(f'Installed rules for a path with {len(path_links)} link(s).')

    while True:
        time.sleep(poll_interval)

        if detect_failure(path_links, base, auth):
            print('Failure detected. Rerouting...')
            # TODO: update path_links with the result of reroute_once(...)
            # path_links = reroute_once(path_links, src_ip, dst_ip, base, auth, app_id)

            if not path_links:
                print('No alternate path found.')
                return

            print(f'Rerouted onto a path with {len(path_links)} link(s).')
        else:
            print('Path still healthy.')

<details>
<summary>Show solution</summary>

```python
def monitor_and_reroute(src_ip, dst_ip, base, auth, app_id, poll_interval=5):
    device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
    remove_rules_by_app_id(device_ids, base, auth, app_id)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    path_links = get_path(src_device, dst_device, base, auth)

    if not path_links:
        print('No path found.')
        return

    install_path_rules(
        path_links, src_ip, dst_ip, src_host_port, dst_host_port, base, auth, app_id,
    )
    print(f'Installed rules for a path with {len(path_links)} link(s).')

    while True:
        time.sleep(poll_interval)

        if detect_failure(path_links, base, auth):
            print('Failure detected. Rerouting...')
            path_links = reroute_once(path_links, src_ip, dst_ip, base, auth, app_id)

            if not path_links:
                print('No alternate path found.')
                return

            print(f'Rerouted onto a path with {len(path_links)} link(s).')
        else:
            print('Path still healthy.')
```

</details>

**Run the app**

Before you run the next cell, bring the `s1-s2` link back up in Mininet:

```text
mininet> link s1 s2 up
```

Then run the cell below. The cell will keep printing `Path still healthy.` every 5 seconds — that is expected.

While it is running (you will see `[*]` next to the cell):

1. in Mininet, run `h1 ping -c 3 h2` and confirm it works
2. in Mininet, run `link s1 s2 down`
3. watch the cell output — it should print `Failure detected. Rerouting...` within a few seconds
4. run `h1 ping -c 3 h2` again and confirm it still works

**To stop the loop**: click the **■ Stop** button in the Jupyter toolbar.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'
app_id = 'org.onosproject.rest'

monitor_and_reroute(src_ip, dst_ip, base, auth, app_id)

## Cleanup

Before moving on, remove the rules this notebook installed and restore `fwd`.

Run the next cell.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
app_id = 'org.onosproject.rest'

device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
removed = remove_rules_by_app_id(device_ids, base, auth, app_id)
print(f'Removed {removed} rule(s) with appId={app_id}.')

Reactivate `fwd` from the ONOS CLI so the network is ready for the next lab:

```text
onos> app activate org.onosproject.fwd
```

Then verify with `pingall` in Mininet.